In [ ]:
import sys
sys.path.append("..") # needed to import dataloader.py

import torch
import torchvision
from dataloader import MultiTaskBrainDataset
from torch.utils.data import DataLoader
from torchinfo import summary

torch.manual_seed(42) # set seed for reproducibility
device = torch.device("cpu")

In [3]:
class PureConvTumorDetectionNetwork(torch.nn.Module):
    """
    A pure convolutional network for tumor detection with a single output.
    
    This network uses only convolutional layers with progressive downsampling,
    followed by global average pooling and a final linear layer to produce
    a single binary output per sample.
    """

    def __init__(self) -> None:
        super(PureConvTumorDetectionNetwork, self).__init__()

        self.convolutional_sequence = torch.nn.Sequential(
            torch.nn.Conv2d(1, 8, 5, stride=2, padding=0),
            torch.nn.ReLU(),
            torch.nn.Conv2d(8, 16, 5, stride=2, padding=0),
            torch.nn.ReLU(),
            torch.nn.Conv2d(16, 32, 5, stride=2, padding=0),
            torch.nn.ReLU(),
            torch.nn.Conv2d(32, 64, 5, stride=2, padding=0),
            torch.nn.ReLU()
        )

        # Global average pooling reduces to (batch_size, 64)
        # Then a linear layer combines the 64 channels into 1 output
        self.final_linear = torch.nn.Linear(64, 1)

    def forward(self, inputs):
        x = self.convolutional_sequence(inputs)

        # Global average pooling: (batch_size, 64, h, w) -> (batch_size, 64)
        x = torch.nn.functional.adaptive_avg_pool2d(x, 1)
        x = x.squeeze(-1).squeeze(-1)

        # Linear combination of the 128 channels: (batch_size, 128) -> (batch_size, 1)
        x = self.final_linear(x)
        return x

In [4]:
model = PureConvTumorDetectionNetwork()
print(summary(model, input_size=(1, 1, 224, 224)))

Layer (type:depth-idx)                   Output Shape              Param #
PureConvTumorDetectionNetwork            [1, 1]                    --
├─Sequential: 1-1                        [1, 64, 11, 11]           --
│    └─Conv2d: 2-1                       [1, 8, 110, 110]          208
│    └─ReLU: 2-2                         [1, 8, 110, 110]          --
│    └─Conv2d: 2-3                       [1, 16, 53, 53]           3,216
│    └─ReLU: 2-4                         [1, 16, 53, 53]           --
│    └─Conv2d: 2-5                       [1, 32, 25, 25]           12,832
│    └─ReLU: 2-6                         [1, 32, 25, 25]           --
│    └─Conv2d: 2-7                       [1, 64, 11, 11]           51,264
│    └─ReLU: 2-8                         [1, 64, 11, 11]           --
├─Linear: 1-2                            [1, 1]                    65
Total params: 67,585
Trainable params: 67,585
Non-trainable params: 0
Total mult-adds (M): 25.77
Input size (MB): 0.20
Forward/backward pass si

In [ ]:
from torch.utils.data import DataLoader
import os
import json

train_data = MultiTaskBrainDataset(
    os.path.join("..", "brisc2025"),
    mode="detection",
    train_or_test="train",
    subset="train",
    augment=True,
    seed=42
)

val_data = MultiTaskBrainDataset(
    os.path.join("..", "brisc2025"),
    mode="detection",
    train_or_test="train",
    subset="val",
    augment=False,
    seed=42
)

train_loader = DataLoader(train_data, batch_size=4, shuffle=True, num_workers=0)
val_loader = DataLoader(val_data, batch_size=4, shuffle=False, num_workers=0)

loss_fn = torch.nn.BCEWithLogitsLoss(pos_weight=torch.tensor([1/3]))
optimizer = torch.optim.Adam(model.parameters())

best_val_acc    = 0.0
min_improvement = 0.000
checkpoint_dir  = os.path.join("..", "results", "detection", "tumor_detection_v2_grant")
os.makedirs(checkpoint_dir, exist_ok=True)

best_model_path = os.path.join(checkpoint_dir, "best_model.pt")
metadata_path   = os.path.join(checkpoint_dir, "training_metadata.json")

training_metadata = {
    "best_val_acc": 0.0,
    "best_epoch":   None,
    "history":      []
}

for epoch in range(100):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in train_loader:
        optimizer.zero_grad()
        predictions = model(inputs)
        loss = loss_fn(predictions, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        preds = (predictions > 0.5).float()

        correct += (preds == labels).sum().item()
        total += labels.numel()

    epoch_loss = running_loss / len(train_loader)
    epoch_acc  = correct / total

    model.eval()

    val_loss    = 0.0
    val_correct = 0
    val_total   = 0

    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            predictions = model(inputs)
            loss = loss_fn(predictions, labels)

            val_loss += loss.item()
            preds = (predictions > 0.5).float()

            val_correct += (preds == labels).sum().item()
            val_total += labels.numel()

    val_loss /= len(val_loader)
    val_acc   = val_correct / val_total

    print(f"Epoch {epoch + 1:03d} | Train Loss: {epoch_loss:.4f} | Train Acc: {epoch_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

    # Record epoch metrics in metadata
    training_metadata["history"].append({
        "epoch":      epoch + 1,
        "train_loss": round(epoch_loss, 6),
        "train_acc":  round(epoch_acc,  6),
        "val_loss":   round(val_loss,   6),
        "val_acc":    round(val_acc,    6),
    })

    # Save best model and update metadata if improved
    if val_acc >= best_val_acc + min_improvement:
        best_val_acc = val_acc
        training_metadata["best_val_acc"] = round(val_acc, 6)
        training_metadata["best_epoch"]   = epoch + 1

        torch.save({
            "epoch":                epoch + 1,
            "model_state_dict":     model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_acc":              val_acc,
            "val_loss":             val_loss,
        }, best_model_path)
        print(f"  -> Best model saved (epoch {epoch+1}, val_acc={val_acc:.4f})")

    # Write metadata JSON after every epoch
    with open(metadata_path, "w") as f:
        json.dump(training_metadata, f, indent=4)

In [ ]:
import os
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

device = torch.device("cpu")

test_data = MultiTaskBrainDataset(
    os.path.join("..", "brisc2025"),
    mode="detection",
    train_or_test="test",
    augment=False,
    seed=42
)
test_loader = DataLoader(test_data, batch_size=4, shuffle=False, num_workers=0)

checkpoint_dir  = os.path.join("..", "results", "detection", "tumor_detection_v2_grant")
best_model_path = os.path.join(checkpoint_dir, "best_model.pt")
metadata_path   = os.path.join(checkpoint_dir, "training_metadata.json")

checkpoint = torch.load(best_model_path, map_location=device)

state_dict = {k.replace("_orig_mod.", ""): v for k, v in checkpoint["model_state_dict"].items()}

model = PureConvTumorDetectionNetwork()
model.load_state_dict(state_dict)
model.to(device)
model.eval()

loss_fn = torch.nn.BCEWithLogitsLoss()

test_loss    = 0.0
test_correct = 0
test_total   = 0
all_predictions = []
all_labels      = []

with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)

        outputs = model(inputs)
        loss    = loss_fn(outputs, labels.float())

        test_loss += loss.item()

        probs = torch.sigmoid(outputs)
        preds = (probs > 0.5).float()

        test_correct += (preds == labels).sum().item()
        test_total   += labels.numel()

        all_predictions.append(preds.cpu())
        all_labels.append(labels.cpu())

test_loss /= len(test_loader)
test_acc   = test_correct / test_total

all_predictions = torch.cat(all_predictions).numpy()
all_labels      = torch.cat(all_labels).numpy()

precision = precision_score(all_labels, all_predictions)
recall    = recall_score(all_labels, all_predictions)
f1        = f1_score(all_labels, all_predictions)
cm        = confusion_matrix(all_labels, all_predictions)

print(f"\nTest Loss:  {test_loss:.4f} | Test Accuracy: {test_acc:.4f}")
print(f"Precision:  {precision:.4f}")
print(f"Recall:     {recall:.4f}")
print(f"F1 Score:   {f1:.4f}")
print(f"\nConfusion Matrix:\n{cm}")

C:\Users\glawl\AppData\Local\Temp\ipykernel_5760\1815286135.py:21: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(best_model_path, map_location=device


Test Loss:  0.0463 | Test Accuracy: 0.9890
Precision:  1.0000
Recall:     0.9872
F1 Score:   0.9936

Confusion Matrix:
[[140   0]
 [ 11 849]]
